# 17. Short-Term vs Long-Term Memory

CrewAI has two kinds of memory:

- **Short-term** = remembers within the **current** run (then forgets).
- **Long-term** = remembers across **different** runs (saved to disk).

## Real-life analogy

- **Short-term memory** is like keeping something **in your head** during one conversation.
- **Long-term memory** is like writing it in a **diary** so you still know it next week.

## The difference

| | Short-term | Long-term |
|---|-----------|-----------|
| Lasts | Only this run | Across many runs |
| Stored where | In memory (RAM) | Saved on disk |
| Good for | Steps within one task | Remembering past sessions |
| Forgets when | The program ends | Never (until you delete it) |

## The good news for beginners

When you set **`memory=True`**, CrewAI turns on **both** short-term and long-term memory for you automatically. You don't have to manage them separately.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from crewai import Agent, Task, Crew, LLM

llm = LLM(model="gpt-4o-mini")

agent = Agent(role="Helper", goal="Help users", backstory="You remember details.", llm=llm)
task = Task(description="Say hello.", expected_output="A hello.", agent=agent)

crew = Crew(
    agents=[agent],
    tasks=[task],
    memory=True,   # turns on BOTH short-term and long-term memory
)

# In a Jupyter notebook, use kickoff_async() with "await".
result = await crew.kickoff_async()
print(result)

Hello!


## When to care about each

- A **one-time** task? Short-term is enough.
- A **personal assistant** that should remember you tomorrow? You need long-term.

Long-term memory is saved to small files on your computer, so it survives after the program closes.

## Saving memory locally and loading it back

Long-term memory is saved to **files on your computer**. We can choose **which folder** those files live in by setting `CREWAI_STORAGE_DIR` to a local folder.

The idea:
- **Run 1** — we tell the agent a fact. CrewAI **saves** it into our folder (the "diary").
- **Run 2** — a brand-new crew points at the **same folder**, **loads** the saved memory, and remembers the fact.

This is how an assistant can still remember you **tomorrow**, even after the program closed.

In [2]:
import os

# Choose a LOCAL folder to keep the memory (like choosing where your diary lives).
os.environ["CREWAI_STORAGE_DIR"] = "./agent_memory"

from dotenv import load_dotenv
load_dotenv()

from crewai import Agent, Task, Crew, LLM

llm = LLM(model="gpt-4o-mini")

agent = Agent(
    role="Helper",
    goal="Remember details about the user",
    backstory="You remember details the user tells you.",
    llm=llm,
)

# RUN 1: tell the agent a fact to remember.
task = Task(
    description="My favourite colour is green. Please remember this about me.",
    expected_output="A short reply saying you will remember it.",
    agent=agent,
)

crew = Crew(agents=[agent], tasks=[task], memory=True)   # memory is SAVED into ./agent_memory

# In a Jupyter notebook, use kickoff_async() with "await".
print(await crew.kickoff_async())
print("\nMemory saved in this folder:", os.path.abspath("./agent_memory"))

I will remember that your favourite colour is green.

Memory saved in this folder: c:\Users\THIRU\OneDrive\Desktop\gen_ai_course\CrewAI_Framework\agent_memory


Now imagine you **close the program and come back later**. We make a **brand-new crew** that points at the **same folder** — it **loads** the saved memory and remembers our fact.

In [3]:
import os

# Point at the SAME folder as before, so the memory is LOADED.
os.environ["CREWAI_STORAGE_DIR"] = "./agent_memory"

from crewai import Agent, Task, Crew, LLM

llm = LLM(model="gpt-4o-mini")

# A brand-new agent and crew (like opening the app again the next day).
agent = Agent(
    role="Helper",
    goal="Remember details about the user",
    backstory="You remember details the user tells you.",
    llm=llm,
)

# RUN 2: ask about the fact from before. The agent reads it from saved memory.
task = Task(
    description="What is my favourite colour? Use what you remember about me.",
    expected_output="My favourite colour.",
    agent=agent,
)

crew = Crew(agents=[agent], tasks=[task], memory=True)   # memory is LOADED from ./agent_memory

print(await crew.kickoff_async())   # should mention green

Your favourite colour is green.


## Key points to remember

- **Short-term** memory lasts only during the current run.
- **Long-term** memory is saved to disk and lasts across runs.
- `memory=True` turns on **both** automatically.
- Short-term = head; long-term = diary.
- Set **`CREWAI_STORAGE_DIR`** to a local folder to choose **where** memory is saved.
- Point a new crew at the **same folder** to **load** old memory and remember past sessions.
- Use long-term when the app should remember the user tomorrow.